
# CSV & Rasters: Multi-dataset Survey with Derivative Products

This example demonstrates the typical workflow for creating a GS file for an AEM survey in its entirety, i.e., the NetCDF file contains all related datasets together, e.g., raw data, processed data, inverted models, and derivative products. Specifically, this survey contains:

1. Minimally processed (raw) AEM data and raw/processed magnetic data provided by SkyTEM
2. Fully processed AEM data used as input to inversion
3. Laterally constrained inverted resistivity models
4. Point-data estimates of bedrock depth derived from the AEM models
5. Interpolated magnetic and bedrock depth grids

Note:
To make the size of this example more managable, some of the input datasets have been downsampled relative to the source files in the data release referenced below.

Dataset Reference: `Minsley, B.J, Bloss, B.R., Hart, D.J., Fitzpatrick, W., Muldoon, M.A., Stewart, E.K., Hunt, R.J., James, S.R., Foks, N.L., and Komiskey, M.J., 2022, Airborne electromagnetic and magnetic survey data, northeast Wisconsin (ver. 1.1, June 2022): U.S. Geological Survey data release, https://doi.org/10.5066/P93SY9LI.`


In [ ]:
import matplotlib.pyplot as plt
from os.path import join
import numpy as np
import gspy
from gspy import Survey
import xarray as xr
from pprint import pprint
import warnings
warnings.filterwarnings('ignore')

## Initialize the Survey



In [ ]:
# Path to example files
data_path = '..//data_files//skytem_csv'

# Survey metadata file
metadata = join(data_path, "data//skytem_survey.yml")

# Establish the Survey
survey = Survey.from_dict(metadata)

## Create a Data Branch



In [ ]:
data_container = survey.gs.add_container('data', **dict(content = "raw and processed data",
                                                        comment = "<extra info goes here>"))

Attach leaves to the data branch



1. Raw Data



In [ ]:
# Import raw AEM data from CSV-format.
# Define input data file and associated metadata file
d_data1 = join(data_path, 'data//skytem_contractor_data.csv')
d_supp1 = join(data_path, 'data//skytem_contractor_data.yml')

# Add the raw AEM data as a tabular dataset,
# pass the EM system from the survey
rd = data_container.gs.add(key='raw_data', data_filename=d_data1, 
                           metadata_file=d_supp1, system=survey.nominal_system)

2. Processed Data



In [ ]:
# Import processed AEM data from CSV-format.
# Define input data file and associated metadata file
d_data2 = join(data_path, 'data//skytem_processed_data.csv')
d_supp2 = join(data_path, 'data//skytem_processed_data.yml')

Example of how systems can be selected and modified to accurately match the processed data



In [ ]:
system = {"skytem_system" : survey["nominal_system"].isel(lm_gate_times=np.s_[1:], 
                                                          hm_gate_times=np.s_[10:]),
          "magnetic_system" : survey["magnetic_system"]}

Add the processed AEM data as a tabular dataset, passing the updated systems



In [ ]:
pd = data_container.gs.add(key='processed_data', data_filename=d_data2, 
                           metadata_file=d_supp2, system=system)

## Create a Models Branch



In [ ]:
# Create a new container for models
model_container = survey.gs.add_container('models', **dict(content = "Inverted models",
                                                          comment = "This is a test"))

3. Inverted Models



In [ ]:
# Import inverted AEM models from CSV-format.
# Define input data file and associated metadata file
m_data3 = join(data_path, 'model//skytem_inverted_models.csv')
m_supp3 = join(data_path, 'model//skytem_inverted_models.yml')

# Add the inverted AEM models as a tabular dataset
mods = model_container.gs.add(key='inverted_models', data_filename=m_data3, 
                              metadata_file=m_supp3)

## Derivative Products



4. Bedrock Picks



Adding bedrock picks to the 'data' branch



In [ ]:
# Import AEM-based estimated of depth to bedrock from CSV-format.
# Define input data file and associated metadata file
d_data4 = join(data_path, 'data//top_dolomite_blocky_lidar.csv')
d_supp4 = join(data_path, 'data//bedrock_picks.yml')

# Add the AEM-based estimated of depth to bedrock as a tabular dataset
bedrock = data_container.gs.add(key='depth_to_bedrock', data_filename=d_data4, 
                                metadata_file=d_supp4)

5. Raster Maps



Create a 3rd container for the derived raaster maps



In [ ]:
derived_maps = survey.gs.add_container('derived_maps', **dict(content = "raster products derived from airborne data and models"))

# Import interpolated bedrock and magnetic maps from TIF-format.
# Define input metadata file (which contains the TIF filenames linked to variable names)
m_supp5 = join(data_path, 'data//magnetics_bedrock_picks.yml')

# Add the interpolated maps as a raster dataset
maps = derived_maps.gs.add(key='maps', metadata_file=m_supp5)

## View the Data Tree



In [ ]:
print(survey.gs.tree)

In [ ]:
print(survey)

## Save to NetCDF file



In [ ]:
d_out = join(data_path, 'skytem.nc')
survey.gs.to_netcdf(d_out)

## Export just one branch to file



The gspy goal is to have the complete survey in a single file. However, we can also save containers or datasets separately.



In [ ]:
data_container.gs.to_netcdf(join(data_path, 'test_datacontainer.nc'))

## Opening a GS NetCDF 



In [ ]:
new_survey = gspy.open_datatree(d_out)['survey']

## Plotting Examples



In [ ]:
plt.figure()
new_survey['data']['raw_data']['height'].plot()
plt.tight_layout()

In [ ]:
pcd = new_survey['data']['processed_data']
plt.figure()
pcd['tx_altitude'].plot()
plt.tight_layout()

In [ ]:
m = new_survey['derived_maps']['maps']
plt.figure()
m['magnetic_tmi'].plot(cmap='jet')
plt.tight_layout()

plt.show()